# Secure & Faithful — Runner Notebook

Runs the whole pipeline. Works **both** in local VS Code (Jupyter) and in Google Colab — it auto-detects which one you're in.

**Run the cells top to bottom, once each.** After the keys cell, everything is automatic.

| Cell | Does |
|---|---|
| 1 | Detect Colab/local, go to the project folder |
| 2 | Install requirements |
| 3 | Enter your 3 API keys |
| 4 | Download datasets |
| 5 | Build the RAG knowledge base |
| 6 | Smoke test (confirm keys + models work) |
| 7 | Run the experiment grid |
| 8 | Show summary table + Pareto plot |

## 1. Locate the project folder (auto-detect Colab vs local)

In [1]:
import os, sys

try:
    import google.colab  # noqa
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT = '/content/drive/MyDrive/AI-Economics'   # your Drive folder
else:
    # Local Google Drive path on Windows. Edit if your drive letter differs.
    PROJECT = r'K:\My Drive\AI-Economics'

os.chdir(PROJECT)
print('IN_COLAB =', IN_COLAB)
print('Working directory:', os.getcwd())
assert os.path.exists('config.yaml'), 'config.yaml not found — is PROJECT correct?'

IN_COLAB = False
Working directory: K:\My Drive\AI-Economics


## 2. Install requirements
First time only (Colab: every new session). Takes 1–3 minutes.

In Colab, if it asks you to **Restart runtime** afterwards, do it, then re-run cell 1, then skip to cell 3.

In [2]:
!pip install -q -r requirements.txt
print('Requirements installed.')

Requirements installed.



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 3. Enter your API keys
- **Local:** if you already filled `.env`, this cell just loads it — nothing to type.
- **Colab:** either add them to the 🔑 *Secrets* panel (names `GROQ_API_KEY`, `OPENAI_API_KEY`, `OPENROUTER_API_KEY`), or you'll be prompted to paste each one.

Keys are kept only in memory for this session; nothing is printed.

In [3]:
import os
try:
    from dotenv import load_dotenv; load_dotenv()  # local .env, if present
except Exception:
    pass

def ensure_key(name):
    if os.getenv(name):
        print(f'{name}: already set'); return
    val = None
    try:
        from google.colab import userdata
        val = userdata.get(name)
    except Exception:
        val = None
    if not val:
        import getpass
        val = getpass.getpass(f'Paste {name}: ')
    os.environ[name] = val
    print(f'{name}: set')

for k in ['GROQ_API_KEY', 'OPENAI_API_KEY', 'OPENROUTER_API_KEY']:
    ensure_key(k)

GROQ_API_KEY: already set
OPENAI_API_KEY: already set
OPENROUTER_API_KEY: already set


## 4. Download & prepare datasets

In [4]:
from src import data_prep
data_prep.prepare_agriculture()
data_prep.prepare_attacks()

c:\Users\Suhail Raazeeth\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[agri] loading KisanVaani/agriculture-qa-english-only …


c:\Users\Suhail Raazeeth\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Suhail Raazeeth\.cache\huggingface\hub\datasets--KisanVaani--agriculture-qa-english-only. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating train split: 100%|██████████| 22615/22615 

  wrote  1500 rows -> kb_corpus.jsonl
  wrote    40 rows -> benign_qa.jsonl
[attacks] loading deepset/prompt-injections …


c:\Users\Suhail Raazeeth\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Suhail Raazeeth\.cache\huggingface\hub\datasets--deepset--prompt-injections. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating test split: 100%|██████████| 116/116 [00:00<00:00, 1681

[attacks] loading domain attacks from domain_attacks.jsonl …
  wrote    40 rows -> attacks.jsonl


## 5. Build the RAG knowledge base (first run downloads a ~90MB embedder)

In [5]:
from src import build_kb
build_kb.build()

[kb] 1500 source documents


c:\Users\Suhail Raazeeth\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Suhail Raazeeth\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00,

[kb] embedding 1517 chunks (first run downloads the model)…
    256/1517
    512/1517
    768/1517
    1024/1517
    1280/1517
    1517/1517
[kb] done. Collection 'agri_kb' has 1517 chunks.
Next:  python -m src.smoke_test


## 6. Smoke test — confirm keys + all models respond

In [6]:
from src import smoke_test
smoke_test.main()

=== Smoke test ===
[ok] keys present for: GROQ_API_KEY, OPENAI_API_KEY, OPENROUTER_API_KEY
[FAIL] groq/llama-3.3-70b-versatile: NotFoundError: litellm.NotFoundError: GroqException - {"error":{"message":"The model `llama-3.3-70b-versatile` does not exist or you do not have access to it.","type":"invalid_request_error","code":"model_not_found"}}

[FAIL] groq/llama-3.1-8b-instant: NotFoundError: litellm.NotFoundError: GroqException - {"error":{"message":"The model `llama-3.1-8b-instant` does not exist or you do not have access to it.","type":"invalid_request_error","code":"model_not_found"}}

[ok] openai/gpt-4o-mini  -> 'OK'
[FAIL] openrouter/mistralai/mistral-7b-instruct: NotFoundError: litellm.NotFoundError: NotFoundError: OpenrouterException - {"error":{"message":"No endpoints found for mistralai/mistral-7b-instruct.","code":404},"user_id":"user_3Ew3qn00RIZaypsoNspOWHv5yfm"}

[rag] testing retrieval + guarded answer on groq/llama-3.3-70b-versatile …


NotFoundError: litellm.NotFoundError: GroqException - {"error":{"message":"The model `llama-3.3-70b-versatile` does not exist or you do not have access to it.","type":"invalid_request_error","code":"model_not_found"}}


## 7. Run the experiment grid
The main run: 4 models × 3 guardrail modes × (benign + attacks). With the default
small sample sizes this takes a few minutes and costs a few cents.
Progress saves to `results/raw_runs.csv` after each model, so it's safe to re-run.

In [ ]:
from src import run_experiments
run_experiments.run()

## 8. Results — summary table + Pareto trade-off plot

In [ ]:
from src import analyze
analyze.main()

import pandas as pd
from IPython.display import Image, display
display(pd.read_csv('results/summary.csv'))
display(Image('results/pareto.png'))